---
title: "Policy Gradient Minimal Examples"
sidebarTitle: "Policy Gradient Minimal Examples"
description: "Tiny finite-state environments that expose what the REINFORCE estimator actually does. Sampled gradients vs analytical expectations, with a baseline-variance comparison."
tag: "Draft"
---

A very small and concrete introduction to policy gradients using the simplest examples possible.

It covers:

1. a single-state, two-action example with informative samples in both directions,
2. a baseline subsection that shows why variance reduction matters,
3. a two-state extension that demonstrates an often-misunderstood property of REINFORCE: actions whose outcomes do not affect the reward receive zero gradient in expectation, even though per-sample updates can be large.

The goal is not realism. The goal is to make the policy-gradient update visually and mathematically transparent.


## Core idea

In supervised learning, we usually have a target output and minimize a loss such as cross-entropy.

In policy gradient methods, we do something different:

- sample an action or trajectory from the current policy,
- evaluate it with a reward,
- increase the log-probability of actions that led to high reward.

The REINFORCE estimator is

$$
\nabla_\theta J(\theta)
\;=\;
\mathbb{E}_{\tau \sim \pi_\theta}
\Bigl[
R(\tau) \, \nabla_\theta \log \pi_\theta(\tau)
\Bigr],
$$

where $\tau$ is the random trajectory and the expectation is over $\tau \sim \pi_\theta$. In an autoregressive or sequential setting the probability of a trajectory factorizes across steps, so a single terminal reward can affect the gradient contributions of multiple earlier decisions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)


## 1. Single-state, two-action example

The smallest possible policy-gradient system.

One state $s$, two actions $a_1, a_2$, parameterized by a single scalar $\theta$:

$$
\pi_\theta(a_1) = \theta,
\qquad
\pi_\theta(a_2) = 1 - \theta,
\qquad
0 < \theta < 1.
$$

Reward:

$$
R(a_1) = +1,
\qquad
R(a_2) = -1.
$$

Both actions produce informative samples (the original $R(a_2) = 0$ choice silences half the updates, which hides the negative-update mechanism we want to see).


### Expected reward

$$
J(\theta)
\;=\;
\mathbb{E}_{A \sim \pi_\theta}\bigl[ R(A) \bigr]
\;=\;
\theta \cdot 1 + (1-\theta) \cdot (-1)
\;=\;
2\theta - 1.
$$

The analytical gradient is $\partial J / \partial \theta = 2$ at every $\theta \in (0,1)$, and the optimum is $\theta^* = 1$.


### REINFORCE update in this example

The policy-gradient estimator is

$$
\nabla_\theta J(\theta)
\;=\;
\mathbb{E}_{A \sim \pi_\theta}
\Bigl[
R(A) \, \nabla_\theta \log \pi_\theta(A)
\Bigr].
$$

Score terms at the two action *values*:

$$
\nabla_\theta \log \pi_\theta(a_1) = \frac{1}{\theta},
\qquad
\nabla_\theta \log \pi_\theta(a_2) = -\frac{1}{1-\theta}.
$$

Per-sample gradient contributions:

- if we sample $A = a_1$, the contribution is $(+1) \cdot (1/\theta) = 1/\theta$ (push $\theta$ up),
- if we sample $A = a_2$, the contribution is $(-1) \cdot (-1/(1-\theta)) = 1/(1-\theta)$ (also push $\theta$ up, by reducing the probability of the bad action).

Both samples now carry signal, and both push $\theta$ in the optimal direction. Taking the expectation:

$$
\mathbb{E}_{A \sim \pi_\theta}\Bigl[ R(A) \, \nabla_\theta \log \pi_\theta(A) \Bigr]
= \theta \cdot \frac{1}{\theta} + (1-\theta) \cdot \frac{1}{1-\theta}
= 2,
$$

which matches the analytical gradient $\partial J / \partial \theta = 2$.


### What to look for in the plots

We visualize three quantities over iterations:

1. $\theta$ itself, the probability of the good action,
2. the sampled gradient $R(A) \nabla_\theta \log \pi_\theta(A)$ vs the analytical expected value $2$,
3. the sampled reward vs the expected return $J(\theta) = 2\theta - 1$.

Expectation: $\theta$ drifts toward $1$, the sampled gradient hovers around $2$ (with variance that grows as $\theta$ approaches the boundaries because $1/\theta$ and $1/(1-\theta)$ explode), and the sampled reward becomes more reliably $+1$.


In [ ]:
theta = 0.4
lr = 0.05
iterations = 50

theta_history = []
grad_history = []                 # sampled R · ∇log π(A)
reward_history = []               # sampled reward
expected_reward_history = []      # J(θ) = 2θ - 1
expected_grad_history = []        # E[R · ∇log π] = 2 (analytical)

for i in range(iterations):
    if np.random.rand() < theta:
        action = 1
        reward = 1
        grad = reward * (1 / theta)
    else:
        action = 2
        reward = -1
        grad = reward * (-1 / (1 - theta))

    expected_reward_history.append(2 * theta - 1)
    expected_grad_history.append(2.0)

    theta += lr * grad
    theta = np.clip(theta, 1e-3, 1 - 1e-3)

    theta_history.append(theta)
    grad_history.append(grad)
    reward_history.append(reward)

plt.figure(figsize=(8, 4))
plt.plot(theta_history)
plt.title(r"Single-state example: parameter $\theta$ over iterations")
plt.xlabel("Iteration")
plt.ylabel(r"$\theta = \pi(a_1)$")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(grad_history, color="#cbd5e1", linewidth=1, label=r"sampled $R \nabla \log \pi(A)$")
plt.plot(expected_grad_history, color="#2563eb", linewidth=2, linestyle="--",
         label=r"expected $\mathbb{E}[R \nabla \log \pi] = 2$")
plt.title("Single-state example: sampled vs expected gradient")
plt.xlabel("Iteration")
plt.ylabel("Gradient value")
plt.legend(loc="upper right")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(reward_history, color="#cbd5e1", linewidth=1, label=r"sampled $R$")
plt.plot(expected_reward_history, color="#2563eb", linewidth=2, linestyle="--",
         label=r"expected $J(\theta) = 2\theta - 1$")
plt.title(r"Single-state example: sampled reward vs expected objective $J(\theta)$")
plt.xlabel("Iteration")
plt.ylabel("Reward")
plt.legend(loc="lower right")
plt.show()


### Interpretation of the single-state plots

The first plot shows the policy parameter climbing toward $1$. With both samples now carrying gradient (positive when $a_1$ is sampled, also positive when $a_2$ is sampled because we want to *reduce* probability of the bad action), the upward push is consistent.

The second plot shows the sampled gradient. The analytical mean is $2$ (dashed line). Per-sample values fluctuate around $2$, and the fluctuation grows as $\theta$ approaches the boundary because the score terms $1/\theta$ and $1/(1-\theta)$ become large. This is the first sign of the central practical issue with REINFORCE: variance.


## 2. Baselines and variance reduction

The variance of the REINFORCE estimator can be reduced without changing its mean by subtracting a baseline $b$ that does not depend on the action:

$$
\nabla_\theta J(\theta)
\;=\;
\mathbb{E}_{A \sim \pi_\theta}
\Bigl[
\bigl( R(A) - b \bigr) \, \nabla_\theta \log \pi_\theta(A)
\Bigr].
$$

The mean is preserved because the score has zero expectation under $\pi_\theta$:

$$
\mathbb{E}_{A \sim \pi_\theta}\bigl[ \nabla_\theta \log \pi_\theta(A) \bigr]
= \sum_{a} \pi_\theta(a) \, \frac{\nabla_\theta \pi_\theta(a)}{\pi_\theta(a)}
= \nabla_\theta \sum_a \pi_\theta(a)
= \nabla_\theta 1
= 0.
$$

So $\mathbb{E}[(R - b) \nabla \log \pi] = \mathbb{E}[R \nabla \log \pi] - b \cdot 0 = \mathbb{E}[R \nabla \log \pi]$. Any constant $b$ gives an unbiased gradient. The variance, however, *does* depend on $b$, and a well-chosen baseline shrinks it.

A common practical choice is $b = J(\theta)$, the current expected return. Below we compare the variance of the sampled gradient with no baseline against the same quantity with $b = J(\theta) = 2\theta - 1$, across a range of $\theta$.


In [ ]:
# Empirical variance of the REINFORCE estimator with and without a baseline,
# for the single-state, two-action example with R(a_1) = +1, R(a_2) = -1.

n_samples = 20000
theta_grid = np.linspace(0.1, 0.9, 17)

var_no_baseline = []
var_with_baseline = []

for theta in theta_grid:
    A = (np.random.rand(n_samples) < theta).astype(int)  # 1 with prob θ
    R = np.where(A == 1, 1.0, -1.0)
    score = np.where(A == 1, 1.0 / theta, -1.0 / (1.0 - theta))

    g_plain = R * score
    g_centered = (R - (2 * theta - 1)) * score

    var_no_baseline.append(g_plain.var())
    var_with_baseline.append(g_centered.var())

plt.figure(figsize=(8, 4))
plt.plot(theta_grid, var_no_baseline, marker="o", label=r"no baseline")
plt.plot(theta_grid, var_with_baseline, marker="s", label=r"baseline $b = J(\theta) = 2\theta - 1$")
plt.title(r"Variance of the sampled gradient $R \nabla \log \pi(A)$ vs $\theta$")
plt.xlabel(r"$\theta$")
plt.ylabel("Empirical variance")
plt.yscale("log")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 3. Two-state extension (and what it does *not* show)

We extend to a tiny trajectory with two decisions:

- one action $A_1$ in state $s_1$,
- one action $A_2$ in state $s_2$,

where each state has its own policy parameter:

$$
\pi_{\theta_1}(a_1 \mid s_1) = \theta_1,
\qquad
\pi_{\theta_2}(a_1 \mid s_2) = \theta_2.
$$

Reward depends only on the action at $s_2$:

$$
R(\tau) =
\begin{cases}
+1, & A_2 = a_1, \\
-1, & A_2 = a_2.
\end{cases}
$$


### Finite state model

![Finite state diagram of the two-state example](images/two-state-fsm.svg)

*Editable Mermaid source: [`images/two-state-fsm.mermaid.md`](images/two-state-fsm.mermaid.md)*

The structure is: one trajectory, two stochastic decisions, one terminal reward. Crucially, the transition $s_1 \to s_2$ is deterministic and the reward is independent of $A_1$.


### What this example does and does not demonstrate

A naive read of REINFORCE on a multi-step trajectory is that "the terminal reward propagates backward and rewards earlier actions". That is true *per sample*, and you will see $\theta_1$ wander in the trace below. But the analytical gradient with respect to $\theta_1$ is exactly zero in this setup:

$$
\frac{\partial J}{\partial \theta_1}
\;=\;
\mathbb{E}\Bigl[ R(\tau) \, \nabla_{\theta_1} \log \pi_{\theta_1}(A_1 \mid s_1) \Bigr]
\;=\;
\theta_2 \cdot \frac{1}{\theta_1} \cdot \theta_1
+ \theta_2 \cdot \frac{-1}{1-\theta_1} \cdot (1-\theta_1)
\;=\; \theta_2 - \theta_2 \;=\; 0.
$$

(Reward is independent of $A_1$, so no learnable signal exists for $\theta_1$.) What the score-function trick guarantees is that *in expectation* the gradient on $\theta_1$ vanishes. Any movement you see in $\theta_1$ over a single training run is high-variance noise that, given enough iterations and the boundary clipping, will eventually park $\theta_1$ at one of the boundaries from drift alone.

So the lesson here is the *opposite* of credit assignment: REINFORCE correctly assigns zero gradient to actions whose outcomes do not affect the reward, even when those actions appear inside a rewarding trajectory. Real credit assignment requires the earlier action to causally change which states (and hence which rewards) are reachable, which would mean replacing the deterministic transition $s_1 \to s_2$ with a transition whose distribution depends on $A_1$. That is left as a follow-up.


### What to look for in the two-state plots

1. $\theta_1$ and $\theta_2$ across iterations,
2. the sampled gradient contributions for each state.

Expected behavior:

- $\theta_2$ moves systematically upward, because $\partial J / \partial \theta_2 = 1$,
- $\theta_1$ does *not* systematically improve, because $\partial J / \partial \theta_1 = 0$. It will random-walk around its initial value, possibly hitting a clip boundary by chance.


In [ ]:
theta1 = 0.5
theta2 = 0.5

lr = 0.05
iterations = 50

theta1_hist = []
theta2_hist = []
grad1_hist = []
grad2_hist = []
reward2_hist = []

# Analytical expectations at each iteration (evaluated at current θ1, θ2):
#   J(θ1, θ2) = E[R] = θ2 · 1 + (1 − θ2) · (-1) = 2 θ2 - 1
#   ∂J/∂θ1 = 0   (reward is independent of the state-1 action)
#   ∂J/∂θ2 = 2
# The score-function identity guarantees E[R · ∇log π(A_1|s_1)] = 0 even when
# per-sample contributions look large.
expected_J_hist = []
expected_g1_hist = []
expected_g2_hist = []

for i in range(iterations):
    if np.random.rand() < theta1:
        a1 = 1
        score1 = 1 / theta1
    else:
        a1 = 2
        score1 = -1 / (1 - theta1)

    if np.random.rand() < theta2:
        a2 = 1
        reward = 1
        score2 = 1 / theta2
    else:
        a2 = 2
        reward = -1
        score2 = -1 / (1 - theta2)

    g1 = reward * score1
    g2 = reward * score2

    expected_J_hist.append(2 * theta2 - 1)
    expected_g1_hist.append(0.0)
    expected_g2_hist.append(2.0)

    theta1 += lr * g1
    theta2 += lr * g2

    theta1 = np.clip(theta1, 1e-3, 1 - 1e-3)
    theta2 = np.clip(theta2, 1e-3, 1 - 1e-3)

    theta1_hist.append(theta1)
    theta2_hist.append(theta2)
    grad1_hist.append(g1)
    grad2_hist.append(g2)
    reward2_hist.append(reward)

plt.figure(figsize=(8, 4))
plt.plot(theta1_hist, label=r"$\theta_1$ (state 1)")
plt.plot(theta2_hist, label=r"$\theta_2$ (state 2)")
plt.title("Two-state example: policy parameters over iterations")
plt.xlabel("Iteration")
plt.ylabel("Parameter value")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(grad1_hist, color="#cbd5e1", linewidth=1, label=r"sampled $g_1$")
plt.plot(expected_g1_hist, color="#2563eb", linewidth=2, linestyle="--",
         label=r"expected $\mathbb{E}[g_1] = 0$")
plt.plot(grad2_hist, color="#fcd34d", linewidth=1, label=r"sampled $g_2$")
plt.plot(expected_g2_hist, color="#dc2626", linewidth=2, linestyle="--",
         label=r"expected $\mathbb{E}[g_2] = 2$")
plt.title("Two-state example: sampled vs expected gradients")
plt.xlabel("Iteration")
plt.ylabel("Gradient value")
plt.legend(loc="upper right", fontsize=9)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(reward2_hist, color="#cbd5e1", linewidth=1, label=r"sampled $R$")
plt.plot(expected_J_hist, color="#2563eb", linewidth=2, linestyle="--",
         label=r"expected $J(\theta_1, \theta_2) = 2\theta_2 - 1$")
plt.title(r"Two-state example: sampled reward vs expected objective $J$")
plt.xlabel("Iteration")
plt.ylabel("Reward")
plt.legend(loc="lower right")
plt.show()


### Interpretation of the two-state plots

$\theta_2$ improves systematically because reward is genuinely tied to the action at $s_2$, and the analytical gradient is $+2$.

$\theta_1$ has analytical gradient $0$. The trace will appear to drift, but the drift is variance, not learning. With enough iterations, drift plus boundary clipping eventually pin $\theta_1$ at one of the extremes. Different random seeds will pin it at different extremes.

The sampled-gradient plot shows the same thing: $g_2$ hovers around its analytical mean of $2$, while $g_1$ takes on large positive and negative values that average to zero.

This is the unbiasedness property of REINFORCE on display: actions whose outcomes do not affect the reward do not gain or lose probability *on average*. To see actual credit assignment, the transition $s_1 \to s_2$ would have to depend on $A_1$, so that earlier actions causally change which rewards are reachable.


## Final remarks

Three conceptual lessons from this notebook:

1. policy gradient is *reward $\times$ score*, not reward alone, and the score depends on the current policy,
2. an additive baseline does not bias the gradient and can substantially reduce its variance,
3. a terminal reward in a multi-step trajectory does *not* automatically reinforce earlier actions in expectation. It does only when the earlier action causally changes which states (and which rewards) are reachable. The two-state example here was designed to make exactly this distinction visible.

A natural next extension would be a two-step example with a stochastic transition $s_1 \to s_{\text{good}} \mid s_{\text{bad}}$ controlled by $A_1$, where credit assignment is genuinely needed.
